Group Memebrs: Andrew Mankin, Ryan Safa

We built a cnn with the full dataset and also a subset of the data.

In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
import cv2
from sklearn.model_selection import train_test_split

In [4]:
df = pd.read_csv("./dataset/labels.csv")

Process the image paths to matplotlib images

In [5]:
def load_image(path):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)

    if img is None:
        print("Bad image:", path)
        return None

    img = cv2.resize(img, (224, 224))
    img = img.astype("float32") / 255.0
    #img = np.stack([img, img, img], axis=-1)
    return img

In [6]:
images = []
labels = []

for path, label in zip(df["image"], df["label"]):
    img = load_image(path)
    if img is None:
        continue
    images.append(img)
    labels.append(label)

X = np.array(images, dtype=np.float32)
y = np.array(labels, dtype=np.float32)

libpng warning: bKGD: invalid
libpng warning: bKGD: invalid
libpng warning: bKGD: invalid
libpng warning: bKGD: invalid
libpng warning: bKGD: invalid
libpng warning: bKGD: invalid
libpng warning: bKGD: invalid


In [7]:
X = np.expand_dims(X, axis=-1)

Now that the data loaded make the train test split for the full dataset

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
# Verify the correct format
print(X_train.shape)
print(X_test.shape)

(1087, 224, 224, 1)
(272, 224, 224, 1)
(102, 224, 224, 1)
(26, 224, 224, 1)


Create the model for training

In [15]:
# Model
model = tf.keras.models.Sequential([
    tf.keras.Input(shape=(224, 224, 1)),
    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(1, activation='sigmoid')  # binary classification
])

In [16]:
# Verify the model is correct
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 222, 222, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │     5,537,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,630,593 (21.48 MB)

 Trainable params: 5,630,593 (21.48 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Compile the model
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# Fit full dataset
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=10,
    batch_size=32
)

Epoch 1/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 27s 777ms/step - accuracy: 0.6375 - loss: 0.6587 - val_accuracy: 0.6397 - val_loss: 0.6239
Epoch 2/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 26s 769ms/step - accuracy: 0.6513 - loss: 0.6243 - val_accuracy: 0.6507 - val_loss: 0.6094
Epoch 3/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 26s 772ms/step - accuracy: 0.6872 - loss: 0.5810 - val_accuracy: 0.6728 - val_loss: 0.5952
Epoch 4/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 26s 771ms/step - accuracy: 0.7305 - loss: 0.5441 - val_accuracy: 0.7169 - val_loss: 0.5393
Epoch 5/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 26s 778ms/step - accuracy: 0.7461 - loss: 0.5216 - val_accuracy: 0.7279 - val_loss: 0.5134
Epoch 6/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 26s 772ms/step - accuracy: 0.7626 - loss: 0.4924 - val_accuracy: 0.7279 - val_loss: 0.5350
Epoch 7/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 27s 790ms/step - accuracy: 0.7691 - loss: 0.4859 - val_accuracy: 0.7463 - val_loss: 0.5025
Epoch 8/10
34/34 ━━━━━━━━━━━━━━━━━━━━ 27s 784ms/step - accuracy: 0.7746 - loss: 0.4825 - val_accu

In [26]:
# Save the model
model.save("full_dataset.keras")

Now to do the subset becuase there are not enough features to reduce

In [19]:
X_train_sub, X_test_sub, y_train_sub, y_test_sub = train_test_split(
    X[:128], y[:128],
    test_size=0.2,
    stratify=y[:128],
    random_state=42
)

In [21]:
# Verify the correct format
print(X_train_sub.shape)
print(X_test_sub.shape)

(102, 224, 224, 1)
(26, 224, 224, 1)


In [22]:
# Subset Model
model_sub = tf.keras.models.Sequential([
    tf.keras.Input(shape=(224, 224, 1)),
    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),
    tf.keras.layers.Conv2D(128, (3,3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2,2)),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(1, activation='sigmoid')  # binary classification
])

In [23]:
# Check if model is correct
model_sub.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 64)             │     5,537,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,630,593 (21.48 MB)

 Trainable params: 5,630,593 (21.48 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
# Compile the model
model_sub.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# fit subset of the dataset
history_sub = model_sub.fit(
    X_train_sub, y_train_sub,
    validation_data=(X_test_sub, y_test_sub),
    epochs=10,
    batch_size=32
)

Epoch 1/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 613ms/step - accuracy: 0.7745 - loss: 0.4418 - val_accuracy: 0.6923 - val_loss: 0.5385
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 603ms/step - accuracy: 0.7843 - loss: 0.4349 - val_accuracy: 0.6923 - val_loss: 0.5365
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 657ms/step - accuracy: 0.7745 - loss: 0.4158 - val_accuracy: 0.7308 - val_loss: 0.5399
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 641ms/step - accuracy: 0.7647 - loss: 0.4135 - val_accuracy: 0.7308 - val_loss: 0.5558
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 606ms/step - accuracy: 0.7647 - loss: 0.4334 - val_accuracy: 0.7308 - val_loss: 0.5582
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 595ms/step - accuracy: 0.7647 - loss: 0.4330 - val_accuracy: 0.6923 - val_loss: 0.5554
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 589ms/step - accuracy: 0.7843 - loss: 0.4180 - val_accuracy: 0.7308 - val_loss: 0.5430
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 594ms/step - accuracy: 0.7941 - loss: 0.3916 - val_accuracy: 0.7308 - val_loss:

In [27]:
# Save the model
model_sub.save("subset.keras")